
Sistema automático de control de asistencia y gestión de ausencias con verificación de justificativos

Objetivo

•	Detectar ausencias en tiempo real, solicitar la razón al empleado, recolectar y validar justificativos (p. ej., certificado médico) y notificar según reglas y plazos definidos.

Alcance

•	Empleados con horarios definidos (turnos).

•	Canales de contacto: WhatsApp/Email.

•	Tipos de ausencia: enfermedad, asuntos personales, trámites, fuerza mayor, otros.

Flujo principal

1.	Detección de ausencia:

•	El sistema compara el horario esperado vs. marcación/ingreso (reloj, app, fichada). Si no hay check in antes de la hora de corte del turno, se dispara el caso de ausencia.

2.	Primer contacto automático:

•	Mensaje al empleado: “No registramos tu ingreso. ¿Cuál es el motivo?” con botones rápidos: [Enfermedad] [Turno cambiado] [Permiso aprobado] [Fuerza mayor] [Otro] + campo de texto opcional.

3.	Clasificación de motivo:

•	Si el motivo es Enfermedad: solicitar certificado médico y fecha de atención, con plazo de carga (p. ej., 24 h).

•	Si el motivo es Permiso aprobado: pedir número/ID de permiso o adjuntar constancia.

•	Otros: recolectar breve explicación o documento de soporte, si aplica.

4.	Subida de justificativo:

•	El empleado adjunta un documento/foto. El sistema extrae datos (OCR) y valida reglas básicas.

5.	Validación del justificativo (reglas):

•	Fecha: debe ser la fecha actual o dentro de X días del evento (configurable).

•	Legibilidad: nombre del empleado, fecha, profesional/centro, firma/sello o equivalente.

•	Originalidad/alteraciones: comprobación heurística (metadatos, detección de ediciones obvias). Cuando falle, derivar a revisión humana.

6.	Decisión y notificaciones:

•	Si es válido: marcar ausencia como justificada. Opción: notificar silenciosamente al empleador o enviar confirmación breve.

•	Si es no válido o falta cargarlo dentro del plazo: enviar aviso al empleado y al empleador; escalar si no hay respuesta tras X horas.




TOKENS

Minimizar llamadas a la API de LLM.
Resultado

~17 llamadas/mes (≈0.8/día).

Supuestos

5% ausencias/día, 22 días hábiles/mes → 110 casos/mes.

90% responde con botones (10% texto libre).

60% presenta certificado.

80% de certificados se validan con reglas (solo 20% requiere LLM).

Mensajería por WhatsApp/Email con plantillas (sin LLM).

Cálculo (simplificado)

Clasificación por texto libre: 110 × 10% = 11

Certificados que requieren LLM: 110 × 60% × 20% = 13

Ahorro por “llamada combinada” (cuando hay texto libre + certificado): ≈7

Total: 11 + 13 − 7 ≈ 17 llamadas/mes

Cómo lograrlo (en una línea)

Botones obligatorios + OCR local + reglas duras (fechas/campos/legibilidad) + 1 sola llamada combinada al LLM solo si hay duda.
Nota

WhatsApp/Email: usar plantillas para 0 llamadas LLM en mensajería.
Escenario intensivo (no recomendado): >400 llamadas/mes.


Con el diseño ultra-ahorrador que te propuse: ~17 llamadas/mes al LLM.
Coste estimado con Gemini es muy bajo (centavos/mes).
Supuestos para el cálculo

17 llamadas/mes.
Por llamada (combinada): ~800 tokens de entrada + ~120 de salida.
Tokens/mes: entrada ≈ 13,600; salida ≈ 2,040; total ≈ 15,640 (~0.016 M).
Coste estimado (precios públicos aprox. a fines de 2024; verifícalos en la página de Google)

Gemini 1.5 Flash (muy barato):
Entrada ≈ 
0.35
/
M
;
s
a
l
i
d
a
≈
0.35/M;salida≈0.53/M
Costo mensual ≈ 0.0136×0.35 + 0.00204×0.53 ≈ $0.006 (< 1 centavo)
Gemini 1.5 Pro (más caro pero aún bajo):
Entrada ≈ 0.37/M ;salida ≈ 7/M;salida≈21/M
Costo mensual ≈ 0.0136×7 + 0.00204×21 ≈ 0.14–0.20

Referencias rápidas

Si duplicaras los tokens por llamada, Pro seguiría < $0.40/mes; Flash seguiría en centavos.
Si usas el escenario “no ultra-ahorrador” del Excel (~0.13 M tokens/mes): Pro ≈ 1–2/mes; Flash ≪ $1.
Recomendaciones para mantener ese coste

1 sola llamada combinada por caso.
Recortar el texto OCR al mínimo útil.
Respuesta del modelo en JSON corto.
Mensajería con plantillas (sin LLM).



In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv


load_dotenv()
    
api_key = os.getenv("GOOGLE_API_KEY")

def generar_mensaje_simple():

    """Versión simplificada con IA y seguimiento de respuesta"""
    
    # Pedir datos básicos
    nombre = input("👤 Nombre del empleado: ").strip()
    horario = input("⏰ Horario esperado (ej: 08:00 AM): ").strip()
    empresa = input("🏢 Nombre de la empresa: ").strip()
    
    # Opción simple de historial
    historial = input("📊 ¿Tiene historial de ausencias? (s/n): ").lower().strip()
    contexto_historial = "Con historial de ausencias" if historial == 's' else "Sin historial de ausencias"
    
    print("\n🔄 Generando mensaje con IA Gemini...")
    
    try:
        # Prompt super simple pero efectivo
        prompt = f"""
        Escribe un mensaje de Recursos Humanos para {nombre} que no llegó a las {horario}.
        Empresa: {empresa}. {contexto_historial}.
        
        Incluye estas 5 opciones con emojis:
        1️⃣ Enfermedad
        2️⃣ Permiso aprobado  
        3️⃣ Cambio de turno
        4️⃣ Fuerza mayor
        5️⃣ Otro motivo
        
        El mensaje debe ser profesional pero empático.
        """
        
        # Llamada simple a la API
        model = genai.GenerativeModel("gemini-2.0-flash")
        respuesta = model.generate_content(prompt)
        
        # Mostrar resultado
        print("\n" + "💬 MENSAJE GENERADO:" + "="*50)
        print(respuesta.text)
        
        # ✅ NUEVO: Sistema de selección de opciones
        print("\n" + "🎯 SISTEMA DE SEGUIMIENTO:" + "="*45)
        print("¿Qué acción deseas realizar?")
        print("1. Enviar mensaje al empleado")
        print("2. Registrar respuesta del empleado")
        print("3. Guardar y salir")
        
        opcion = input("\nSelecciona una opción (1-3): ").strip()
        
        if opcion == "1":
            print(f"\n✅ Mensaje enviado a {nombre} por correo/WhatsApp")
            print("📧 Modo simulación: Mensaje enviado exitosamente!")
            
        elif opcion == "2":
            print(f"\n📝 Registrando respuesta de {nombre}...")
            print("\nOpciones de respuesta:")
            print("1. Enfermedad")
            print("2. Permiso aprobado")
            print("3. Cambio de turno")
            print("4. Fuerza mayor")
            print("5. Otro motivo")
            
            respuesta_empleado = input("\nIngresa el número de la respuesta del empleado (1-5): ").strip()
            
            # Mapear opciones
            opciones = {
                "1": "Enfermedad",
                "2": "Permiso aprobado", 
                "3": "Cambio de turno",
                "4": "Fuerza mayor",
                "5": "Otro motivo"
            }
            
            if respuesta_empleado in opciones:
                motivo = opciones[respuesta_empleado]
                print(f"\n✅ Respuesta registrada: {motivo}")
                print(f"📋 Se ha registrado en el sistema para {nombre}")
            else:
                print("❌ Opción no válida")
                
        elif opcion == "3":
            # Guardar archivo
            filename = f"mensaje_{nombre}.txt"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(respuesta.text)
            print(f"✅ Mensaje guardado como '{filename}'")
            
        else:
            print("❌ Opción no válida. Saliendo...")
            
    except Exception as e:
        # Mensaje de emergencia si falla la IA
        print(f"❌ Error: {e}")
        print(f"""
Estimado/a {nombre},

No registramos su ingreso a las {horario}. Por favor, indique el motivo:

1️⃣ Enfermedad
2️⃣ Permiso aprobado
3️⃣ Cambio de turno  
4️⃣ Fuerza mayor
5️⃣ Otro motivo

Responda con el número correspondiente.

Recursos Humanos - {empresa}
        """)

# Ejecutar directamente
if __name__ == "__main__":
    generar_mensaje_simple()